Modules:

In [1]:
import pandas as pd
import os
from tqdm import tqdm
from dotenv import load_dotenv

Config:

In [2]:
# --- CONFIGURATION ---
# Prompt Technique Toggles
EXAMPLES_ENABLED = False    # 3-shot examples
REASONING_ENABLED = False   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
ROLE_NR = 1                   # Choose the system prompt (role): default = 1
ROLE = {1: "an expert in ESG investment", 
         2: "an expert in analyzing text readability", 
         3: "a professor of linguistics",
         4: "an academic expert in ESG (Environmental, Social, and Governance) analysis"}[ROLE_NR]

# --- FILES & MODEL ---
MODEL = "GPT5"
OUTPUT_FILE = f"PoC readability_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES_ENABLED else ""}{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.csv"
# OUTPUT_FILE = "PoC read ground truth.csv"     # for ground truth average scores calculation
INPUT_FILE = OUTPUT_FILE

# The 12 Shimamura et al. readability criteria
SUBCRITERIA = [
    "A-01", "A-02", "B-01", "B-02", "B-03", 
    "C-01", "C-02", "C-03", "C-04", "C-05", "C-06", "D-01"
]

# Mapping to main criteria for calculating the averages
PARENT_MAP = {
    "A-01": "A", "A-02": "A",
    "B-01": "B", "B-02": "B", "B-03": "B",
    "C-01": "C", "C-02": "C", "C-03": "C", "C-04": "C", "C-05": "C", "C-06": "C",
    "D-01": "D"
}

# Configuration for calculating a general readability score:
# Four scores get calculated: one is just the average of the three main criteria averages, one is the weighted average of the three main criteria averages,
# one is the weighted average of all subcriteria scores, one is the general average of all subcriteria scores.
WEIGHTS_PER_SUBCRIT = {
    "A-01": 1, "A-02": 1,
    "B-01": 1, "B-02": 1, "B-03": 1,
    "C-01": 1, "C-02": 1, "C-03": 1, "C-04": 1, "C-05": 1, "C-06": 1,
    "D-01": 1
}
WEIGHTS_PER_MAINCRIT = {
    "A": 1, "B": 1, "C": 1, "D": 1
}

# Average scores calculation:

In [3]:
def main():
    # 1. Verification of required files
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    # 2. Load Data
    df = pd.read_csv(INPUT_FILE)
    print(INPUT_FILE, "opened succesfully")
    # Expected columns: Company, Date, Link, Text
    text_col = "Text" 

    # 3. Initialize Columns
    # Initialize Word_Count column
    if "Word_Count" not in df.columns: df["Word_Count"] = None

    # Initialize Averages
    for main_crit in ["Avg_A", "Avg_B", "Avg_C", "Avg_D"]:
        if main_crit not in df.columns: df[main_crit] = None

    for average_score in ["Average_Main_Crit","Average_Subcrit", "Weighted_Main_Crit", "Weighted_Subcrit"]:
        if average_score not in df.columns: df[average_score] = None

    # Calculate range
    end_index = len(df)

    print(f"Processing posts from index 0 to {end_index}...")

    # 4. Processing Loop
    for index in tqdm(range(end_index), desc="Calculating averages"):
        post_content = str(df.at[index, text_col])
        
         # Skip if not successfully processed yet
        if pd.notna(df.at[index, 'Gunning_Fog']) and "ERROR" not in str(df.at[index, 'Gunning_Fog']):

            # 1. Calculate word count
            df.at[index, "Word_Count"] = len(post_content.split())

            # 3. Map Scores and Calculate Averages per Main Criteria
            crit_scores = {"A": [], "B": [], "C": [], "D": []} # for the average score per criteria
            average_maincrit = [] # for the average total score (using main criteria)
            weighted_subcrit = [] # for the weighted average total score (using subcriteria)
            weighted_maincrit = [] # for the weighted average total score (using main criteria)
            subcrit_weights = 0
            total_scores = 0
            n_scores = 0

            for sub in SUBCRITERIA:
                clean_sub = sub.replace("-", "_")
                score_val = df.at[index, f"{clean_sub}_score"]

                # Prep for Averages (if numeric)
                if score_val and pd.notna(score_val):
                    try:
                        num_score = float(score_val)
                        parent = PARENT_MAP[sub]
                        crit_scores[parent].append(num_score)
                        weighted_subcrit.append(num_score * WEIGHTS_PER_SUBCRIT[sub])
                        subcrit_weights += WEIGHTS_PER_SUBCRIT[sub]
                        total_scores += num_score
                        n_scores += 1
                    except ValueError:
                        pass 
            # 4. Compute and save Averages
            for cat_prefix, scores in crit_scores.items():
                if scores:
                    av = sum(scores) / len(scores)
                    average_maincrit.append(av)
                    weighted_maincrit.append(av * WEIGHTS_PER_MAINCRIT[cat_prefix])
                    df.at[index, f"Avg_{cat_prefix}"] = av

            df.at[index, "Average_Main_Crit"] = sum(average_maincrit) / len(average_maincrit) if average_maincrit else 0
            df.at[index, "Average_Subcrit"] = total_scores / n_scores

            df.at[index, "Weighted_Main_Crit"] = sum(weighted_maincrit) / sum(WEIGHTS_PER_MAINCRIT.values()) if weighted_maincrit else 0
            df.at[index, "Weighted_Subcrit"] = sum(weighted_subcrit) / subcrit_weights if weighted_subcrit else 0

    # Final Save
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

PoC readability_GPT5_role1.csv opened succesfully
Processing posts from index 0 to 20...


Calculating averages: 100%|██████████| 20/20 [00:00<00:00, 1649.09it/s]


Processing complete! Results saved to PoC readability_GPT5_role1.csv
